In [1]:
# ==========================================
# CELL 1: Imports & Correct Data Loading
# ==========================================
import os
import joblib
import numpy as np
import pandas as pd

# Missing sklearn imports
from sklearn.ensemble import (
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.metrics import r2_score, mean_absolute_error

# Dynamic Project Root Finder
current_dir = os.getcwd()
while not os.path.exists(os.path.join(current_dir, "data")):
    parent_dir = os.path.dirname(current_dir)
    if parent_dir == current_dir:
        raise FileNotFoundError("Could not locate project root directory!")
    current_dir = parent_dir

PROJECT_ROOT = current_dir

# Correct paths pointing directly to data/processed/
TRAIN_CSV = os.path.join(PROJECT_ROOT, "data", "processed", "train_engineered.csv")
TEST_CSV = os.path.join(PROJECT_ROOT, "data", "processed", "test_engineered.csv")

print(f"Loading Train Data from: {TRAIN_CSV}")
print(f"Loading Test Data from : {TEST_CSV}")

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

# Define 9 non-redundant features and target
FEATURES = [
    "Bedroom",
    "Bath",
    "Area_sqft",
    "Area_per_bedroom",
    "Bath_bedroom_ratio",
    "Location_median_ppsf",
    "Type_target_enc",
    "City_target_enc",
    "Source_OLX",
]
TARGET = "Log_price"

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

actual_train_pkr = train_df["Price_pkr"]
actual_test_pkr = test_df["Price_pkr"]

print("Data Loaded Successfully!")
print(f"Train Shape: {X_train.shape} | Test Shape: {X_test.shape}")

Loading Train Data from: c:\Users\DELL\Desktop\Project\Real-Estate-Intelligence-System\data\processed\train_engineered.csv
Loading Test Data from : c:\Users\DELL\Desktop\Project\Real-Estate-Intelligence-System\data\processed\test_engineered.csv
Data Loaded Successfully!
Train Shape: (2484, 9) | Test Shape: (621, 9)


In [2]:
# ==========================================
# CELL 2: Train Ensemble Base Models
# ==========================================
print("Training Ensemble Models (Gradient Boosting, HistGBR, Random Forest)...")

# 1. Gradient Boosting
m_gbr = GradientBoostingRegressor(
    n_estimators=180, max_depth=5, learning_rate=0.08, random_state=42
)
m_gbr.fit(X_train, y_train)

# 2. Histogram Gradient Boosting
m_hgbr = HistGradientBoostingRegressor(
    max_iter=300, max_depth=6, learning_rate=0.04, random_state=42
)
m_hgbr.fit(X_train, y_train)

# 3. Random Forest
m_rf = RandomForestRegressor(
    n_estimators=200, max_depth=15, random_state=42, n_jobs=-1
)
m_rf.fit(X_train, y_train)

print("All models trained successfully!")

Training Ensemble Models (Gradient Boosting, HistGBR, Random Forest)...
All models trained successfully!


In [3]:
# ==========================================
# CELL 3: Ensemble Prediction & Evaluation
# ==========================================
# Weighted average of log predictions
test_preds_log = (
    (0.50 * m_gbr.predict(X_test))
    + (0.30 * m_hgbr.predict(X_test))
    + (0.20 * m_rf.predict(X_test))
)

# Convert log scale predictions back to actual PKR prices
test_preds_pkr = np.exp(test_preds_log)

# Calculate accuracy metrics
r2 = r2_score(y_test, test_preds_log)
mae = mean_absolute_error(actual_test_pkr, test_preds_pkr)
mape = np.mean(np.abs((actual_test_pkr - test_preds_pkr) / actual_test_pkr)) * 100

errors_pct = np.abs((actual_test_pkr - test_preds_pkr) / actual_test_pkr) * 100
within_10 = (errors_pct <= 10).mean() * 100
within_20 = (errors_pct <= 20).mean() * 100

print("\n" + "=" * 50)
print("          ENSEMBLE MODEL ACCURACY RESULTS          ")
print("=" * 50)
print(f"Test R² Score (Variance Explained) : {r2:.4f}")
print(f"Test MAE (Mean Error in PKR)      : PKR {mae:,.2f}")
print(f"Test MAPE (Percentage Error)      : {mape:.2f}%")
print("-" * 50)
print(f"Properties within ±10% Margin     : {within_10:.2f}%")
print(f"Properties within ±20% Margin     : {within_20:.2f}%")
print("=" * 50)


          ENSEMBLE MODEL ACCURACY RESULTS          
Test R² Score (Variance Explained) : 0.8443
Test MAE (Mean Error in PKR)      : PKR 20,597,197.90
Test MAPE (Percentage Error)      : 30.27%
--------------------------------------------------
Properties within ±10% Margin     : 31.24%
Properties within ±20% Margin     : 52.98%


In [4]:
# ==========================================
# CELL 4: Save Model Artifact
# ==========================================
model_dir = os.path.join(PROJECT_ROOT, "models")
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "property_price_model.pkl")

pipeline_artifact = {
    "gbr": m_gbr,
    "hgbr": m_hgbr,
    "rf": m_rf,
    "features": FEATURES,
    "weights": [0.50, 0.30, 0.20],
}

joblib.dump(pipeline_artifact, model_path)
print(f"Successfully saved model artifact to '{model_path}'!")

Successfully saved model artifact to 'c:\Users\DELL\Desktop\Project\Real-Estate-Intelligence-System\models\property_price_model.pkl'!
